# Exploratory Supervised Cross-Sensor Transfer Learning

This notebook is kept as exploratory code only. It is not part of the final experiments reported in the thesis, whose retained strategies are baseline selection, online data augmentation, custom loss, DA+CL, and multimodal self-supervised learning.

The configuration below is aligned with the final DA+CL hyperparameters to avoid contradictions, but the transfer-learning experiment should not be cited as a final result of the thesis.

In [ ]:
import copy
import gc
import os
import random
import re
import warnings
from collections import Counter
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data" / "raw").exists():
    raise RuntimeError("Could not locate the project root containing data/raw.")

MODELS_TO_RUN = ["ViT"]
TASKS_TO_RUN = ["11f", "2f"]
SENSORS_TO_RUN = ["clinical", "dermoscopic"]
TRANSFER_DIRECTIONS = [
    {"source_sensor": "clinical", "target_sensor": "dermoscopic", "direction": "clinical_to_dermoscopic"},
    {"source_sensor": "dermoscopic", "target_sensor": "clinical", "direction": "dermoscopic_to_clinical"},
]

# Winning custom-loss configurations selected by validation objective in the previous notebooks.
# They are reused here for both transfer directions and both sensors.
WINNING_TRANSFER_FULL_CONFIGS = {
    "11f": {"model": "ViT", "lr": 3e-5, "batch_size": 128, "beta": 0.999},
    "2f": {"model": "ViT", "lr": 3e-5, "batch_size": 64, "beta": 0.99},
}

SELECTED_RUN_CONFIGS = [
    {"task": task, "sensor": sensor, **WINNING_TRANSFER_FULL_CONFIGS[task]}
    for task in TASKS_TO_RUN
    for sensor in SENSORS_TO_RUN
]

EARLY_STOPPING_PATIENCE_BY_MODEL = {"ViT": 3}
N_RUNS_BY_MODEL = {"ViT": 3}

EPOCHS = 100
ALPHA = 0.40
BASE_SEED = 42
RESUME = True

NUM_WORKERS = 0 if os.name == "nt" else min(4, os.cpu_count() or 1)
PIN_MEMORY = torch.cuda.is_available()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EXPERIMENT_ROOT = PROJECT_ROOT / "Experiment_5" / "transfer_learning_full"
EXPERIMENT_NAME = "online_aug_custom_loss"
MODELS_ROOT = EXPERIMENT_ROOT / "models" / EXPERIMENT_NAME
REPORTS_ROOT = EXPERIMENT_ROOT / "reports" / EXPERIMENT_NAME
RUNS_ROOT = EXPERIMENT_ROOT / "runs" / EXPERIMENT_NAME
PLOTS_ROOT = EXPERIMENT_ROOT / "plots" / EXPERIMENT_NAME

for path_obj in [MODELS_ROOT, REPORTS_ROOT, RUNS_ROOT, PLOTS_ROOT]:
    path_obj.mkdir(parents=True, exist_ok=True)

DATASET_ROOTS = {
    ("11f", "clinical"): PROJECT_ROOT / "data" / "dataset_11f" / "clinical",
    ("11f", "dermoscopic"): PROJECT_ROOT / "data" / "dataset_11f" / "dermoscopic",
    ("2f", "clinical"): PROJECT_ROOT / "data" / "dataset_2f" / "clinical",
    ("2f", "dermoscopic"): PROJECT_ROOT / "data" / "dataset_2f" / "dermoscopic",
}

EXPECTED_CLASSES = {"11f": 11, "2f": 2}

PAPER_HFLIP_P = 0.50
PAPER_VFLIP_P = 0.50
PAPER_ROTATION_RANGE = (-15, 15)
PAPER_SCALE = (0.90, 1.10)
PAPER_SHEAR = (-10, 10)
PAPER_JITTER = (0.20, 0.20, 0.05, 0.05)
TRANSFORM_TYPES = ["hflip", "vflip", "rotation", "scale", "shear", "jitter"]

OFFLINE_AUG_RECIPES = []
for mask in range(1, 2 ** len(TRANSFORM_TYPES)):
    active = [TRANSFORM_TYPES[idx] for idx in range(len(TRANSFORM_TYPES)) if (mask >> idx) & 1]
    active_set = set(active)
    recipe_slug = "_".join(active)
    OFFLINE_AUG_RECIPES.append({
        "recipe_name": recipe_slug,
        "hflip_p": PAPER_HFLIP_P if "hflip" in active_set else 0.0,
        "vflip_p": PAPER_VFLIP_P if "vflip" in active_set else 0.0,
        "rotation_range": PAPER_ROTATION_RANGE if "rotation" in active_set else (0, 0),
        "scale": PAPER_SCALE if "scale" in active_set else (1.0, 1.0),
        "shear": PAPER_SHEAR if "shear" in active_set else (0.0, 0.0),
        "jitter": PAPER_JITTER if "jitter" in active_set else (0.0, 0.0, 0.0, 0.0),
        "translate": 0.0,
        "transform_types": "+".join(active),
    })

RECIPE_CFG_MAP = {recipe["recipe_name"]: recipe for recipe in OFFLINE_AUG_RECIPES}
REFERENCE_AUG_ROOT = PROJECT_ROOT / "Experiment_2"
REFERENCE_SELECTION_METRIC = "val_objective_auc_eo"
ONLINE_EXPERIMENT_NAME = "online_class_aware_custom_loss_transfer"
SAMPLER_GAMMA = 0.5
AUG_MULTIPLICITY_CAP = 6
AUG_MULTIPLICITY_EXPONENT = 0.5
DEFAULT_CUSTOM_LOSS_BETA = 0.999

HF_TOKEN = os.getenv("HF_TOKEN", "").strip()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception:
        pass
else:
    warnings.filterwarnings("ignore", message=r"You are sending unauthenticated requests to the HF Hub.*")

print("Project root:", PROJECT_ROOT)
print("Experiment root:", EXPERIMENT_ROOT)
print("Results namespace:", EXPERIMENT_NAME)
print("Reference augmentation root:", REFERENCE_AUG_ROOT)
print("Device:", DEVICE)
print("Winning full-transfer configs:", WINNING_TRANSFER_FULL_CONFIGS)
print("WeightedRandomSampler gamma:", SAMPLER_GAMMA)
print("Augmentation multiplicity rule:", "repeat_c = clamp(round((max_count / count_c) ** 0.5), 1, 6)")

In [ ]:
metadata_path = Path("data/raw/MILK10k_Training_Metadata.csv")
raw_metadata_df = pd.read_csv(
    metadata_path,
    usecols=["isic_id", "skin_tone_class", "sex", "age_approx"],
).copy()

metadata_df = pd.DataFrame({
    "isic_id": raw_metadata_df["isic_id"],
    "sex": raw_metadata_df["sex"].fillna("unknown").astype(str).str.strip().str.lower(),
    "skin_tone": raw_metadata_df["skin_tone_class"].fillna(-1).astype(int),
})

metadata_df["age"] = raw_metadata_df["age_approx"].apply(
    lambda value: "unknown" if pd.isna(value) else int(value)
)

SEX_TO_ID = {"male": 0, "female": 1, "unknown": 2}

isic_to_skin = dict(zip(metadata_df["isic_id"], metadata_df["skin_tone"]))
isic_to_sex = dict(zip(metadata_df["isic_id"], metadata_df["sex"]))
isic_to_age = dict(zip(metadata_df["isic_id"], metadata_df["age"]))


def normalize_sex(value: str) -> str:
    value = (value or "unknown").strip().lower()
    return value if value in SEX_TO_ID else "unknown"


def normalize_age(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return "unknown"
    if isinstance(value, str):
        stripped = value.strip().lower()
        if stripped == "unknown" or stripped == "":
            return "unknown"
        try:
            return int(float(stripped))
        except Exception:
            return "unknown"
    try:
        return int(value)
    except Exception:
        return "unknown"


def format_lr(lr: float) -> str:
    return f"{lr:.0e}".replace("e-0", "e-").replace("e+0", "e+")


def safe_name(name: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "_", str(name).strip().lower()).strip("_")


print("metadata_df:", metadata_df.shape)
metadata_df.head()


In [ ]:
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


IMAGE_SIZE = 224

DATA_TRANSFORM = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
])


def load_datasets(task: str, sensor: str):
    dataset_root = DATASET_ROOTS[(task, sensor)]

    train_dataset = ImageFolder(dataset_root / "train")
    val_dataset = ImageFolder(dataset_root / "validation", transform=DATA_TRANSFORM)
    test_dataset = ImageFolder(dataset_root / "test", transform=DATA_TRANSFORM)

    class_names = train_dataset.classes
    if len(class_names) != EXPECTED_CLASSES[task]:
        raise RuntimeError(f"Expected {EXPECTED_CLASSES[task]} classes for {task}, but found {len(class_names)} in {dataset_root}.")

    if val_dataset.classes != class_names or test_dataset.classes != class_names:
        raise RuntimeError("Class folders are not consistent across train, validation, and test.")

    return train_dataset, val_dataset, test_dataset, class_names


def make_eval_dataloaders(val_dataset, test_dataset, batch_size: int):
    persistent = NUM_WORKERS > 0
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=persistent)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=persistent)
    return val_loader, test_loader


for task in TASKS_TO_RUN:
    for sensor in SENSORS_TO_RUN:
        train_data, val_data, test_data, class_names = load_datasets(task, sensor)
        print(f"{task} | {sensor}: train={len(train_data)} validation={len(val_data)} test={len(test_data)} classes={len(class_names)}")

In [ ]:
def format_lr(lr):
    return f"{lr:.0e}".replace("e-0", "e-")


def format_float_tag(value: float) -> str:
    return f"{value:g}".replace(".", "p")


def build_recipe_augment(recipe):
    ops = []
    if recipe.get("hflip_p", 0) > 0:
        ops.append(transforms.RandomHorizontalFlip(p=float(recipe["hflip_p"])))
    if recipe.get("vflip_p", 0) > 0:
        ops.append(transforms.RandomVerticalFlip(p=float(recipe["vflip_p"])))

    rotation_range = recipe.get("rotation_range", (0.0, 0.0))
    if isinstance(rotation_range, (int, float)):
        rotation_range = (0.0, float(rotation_range))
    rotation_range = tuple(rotation_range)
    if rotation_range != (0.0, 0.0):
        ops.append(transforms.RandomRotation(rotation_range))

    translate = float(recipe.get("translate", 0.0))
    scale = tuple(recipe.get("scale", (1.0, 1.0)))
    shear = recipe.get("shear", (0.0, 0.0))
    if isinstance(shear, (int, float)):
        shear = (-float(shear), float(shear))
    shear = tuple(shear)
    use_affine = (translate > 0) or (scale != (1.0, 1.0)) or (shear != (0.0, 0.0))
    if use_affine:
        ops.append(transforms.RandomAffine(degrees=0, translate=(translate, translate) if translate > 0 else None, scale=scale if scale != (1.0, 1.0) else None, shear=shear if shear != (0.0, 0.0) else None))

    brightness, contrast, saturation, hue = recipe.get("jitter", (0.0, 0.0, 0.0, 0.0))
    if any(value > 0 for value in [brightness, contrast, saturation, hue]):
        ops.append(transforms.ColorJitter(brightness=brightness, contrast=contrast, saturation=saturation, hue=hue))

    return transforms.Compose(ops) if ops else transforms.Lambda(lambda image: image)


def read_csv_if_exists(csv_path):
    csv_path = Path(csv_path)
    if not csv_path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(csv_path)
    except Exception:
        return pd.DataFrame()


def baseline_offline_dir(task, sensor="dermoscopic"):
    return REFERENCE_AUG_ROOT / "runs" / "ViT" / task / sensor / "offline"


def recipe_score_from_runs(recipe_dir):
    experiments_csv = recipe_dir / "experiments.csv"
    averages_csv = recipe_dir / "averages.csv"
    exp_df = read_csv_if_exists(experiments_csv)
    if not exp_df.empty and REFERENCE_SELECTION_METRIC in exp_df.columns:
        scored = exp_df.copy()
        if "status" in scored.columns:
            scored = scored[scored["status"] == "ok"]
        scored[REFERENCE_SELECTION_METRIC] = pd.to_numeric(scored[REFERENCE_SELECTION_METRIC], errors="coerce")
        scored = scored.dropna(subset=[REFERENCE_SELECTION_METRIC])
        if not scored.empty:
            return float(scored[REFERENCE_SELECTION_METRIC].mean()), int(len(scored)), "experiments.csv"

    avg_df = read_csv_if_exists(averages_csv)
    if not avg_df.empty and REFERENCE_SELECTION_METRIC in avg_df.columns:
        scored = avg_df.copy()
        scored[REFERENCE_SELECTION_METRIC] = pd.to_numeric(scored[REFERENCE_SELECTION_METRIC], errors="coerce")
        scored = scored.dropna(subset=[REFERENCE_SELECTION_METRIC])
        if not scored.empty:
            n_runs = int(pd.to_numeric(scored.get("n_runs_completed", pd.Series([np.nan])), errors="coerce").max()) if "n_runs_completed" in scored.columns else np.nan
            return float(scored[REFERENCE_SELECTION_METRIC].max()), n_runs, "averages.csv"
    return None, 0, "none"


def load_top_recipes_from_baseline(task, sensor="dermoscopic", n_top=10):
    offline_dir = baseline_offline_dir(task, sensor)
    if not offline_dir.exists():
        raise FileNotFoundError(f"Experiment_2 offline runs not found: {offline_dir}")
    rows = []
    for recipe_dir in sorted([path for path in offline_dir.iterdir() if path.is_dir()]):
        score, n_runs, score_source = recipe_score_from_runs(recipe_dir)
        if score is None:
            continue
        rows.append({"task": task, "sensor": sensor, "recipe_name": recipe_dir.name, "score": float(score), "selection_metric": REFERENCE_SELECTION_METRIC, "score_source": score_source, "n_runs_used": int(n_runs) if pd.notna(n_runs) else np.nan})
    if not rows:
        raise RuntimeError(f"No valid offline recipes found in Experiment_2 for {task}/{sensor} using {REFERENCE_SELECTION_METRIC}.")
    top_df = pd.DataFrame(rows).sort_values("score", ascending=False).head(n_top).reset_index(drop=True)
    top_recipes = [RECIPE_CFG_MAP[name] for name in top_df["recipe_name"].tolist() if name in RECIPE_CFG_MAP]
    return top_df, top_recipes


TOP10_RECIPES_INFO = {}
TOP10_RECIPES_BY_TASK_SENSOR = {}
for task in TASKS_TO_RUN:
    top_df, top_recipes = load_top_recipes_from_baseline(task=task, sensor="dermoscopic", n_top=10)
    TOP10_RECIPES_INFO[(task, "dermoscopic_reference")] = top_df
    for sensor in SENSORS_TO_RUN:
        TOP10_RECIPES_BY_TASK_SENSOR[(task, sensor)] = top_recipes
    print(f"Top 10 recipes from Experiment_2 ({task}/dermoscopic/offline), reused online for both sensors:")
    display(top_df)


def extract_source_stem(path):
    stem = Path(path).stem
    parts = stem.split("__")
    if len(parts) >= 3 and parts[0].startswith("exp"):
        return parts[1]
    return stem


ISIC_TO_CLASS11_BY_SENSOR = {}
for sensor in SENSORS_TO_RUN:
    mapping = {}
    train_dir = DATASET_ROOTS[("11f", sensor)] / "train"
    for class_dir in sorted([path for path in train_dir.iterdir() if path.is_dir()]):
        for image_path in class_dir.glob("*.jpg"):
            mapping[image_path.stem] = class_dir.name
    ISIC_TO_CLASS11_BY_SENSOR[sensor] = mapping


def resolve_policy_class(task, sensor, path, class_name):
    if task == "11f":
        return class_name
    return ISIC_TO_CLASS11_BY_SENSOR.get(sensor, {}).get(extract_source_stem(path), class_name)


def compute_policy_class_counts(train_dir, task, sensor):
    base_ds = ImageFolder(root=train_dir)
    counts = Counter()
    for path, y in base_ds.samples:
        class_name = base_ds.classes[int(y)]
        policy_class = resolve_policy_class(task, sensor, path, class_name)
        counts[policy_class] += 1
    return dict(counts)


def compute_class_aware_repeat_map(train_dir, task, sensor, exponent=AUG_MULTIPLICITY_EXPONENT, cap=AUG_MULTIPLICITY_CAP):
    class_counts = compute_policy_class_counts(train_dir, task, sensor)
    if not class_counts:
        raise RuntimeError("No class counts found for class-aware augmentation policy.")
    max_count = max(class_counts.values())
    repeat_map = {}
    for class_name, count in class_counts.items():
        ratio = max_count / max(int(count), 1)
        repeat_value = int(round(ratio ** float(exponent)))
        repeat_map[class_name] = int(max(1, min(cap, repeat_value)))
    return class_counts, repeat_map


CLASS_AWARE_COUNTS = {}
CLASS_AWARE_REPEAT_MAPS = {}
for task in TASKS_TO_RUN:
    for sensor in SENSORS_TO_RUN:
        train_dir = DATASET_ROOTS[(task, sensor)] / "train"
        class_counts, repeat_map = compute_class_aware_repeat_map(train_dir, task, sensor)
        CLASS_AWARE_COUNTS[(task, sensor)] = class_counts
        CLASS_AWARE_REPEAT_MAPS[(task, sensor)] = repeat_map
        print(f"Class-aware augmentation policy for {task} | {sensor}")
        display(pd.DataFrame([{"class": cls, "train_count": int(class_counts[cls]), "repeat_factor": int(repeat_map[cls])} for cls in sorted(repeat_map.keys())]))


class OnlineClassAwareTop10Dataset(Dataset):
    def __init__(self, train_dir, top_recipes, task, sensor):
        self.base_ds = ImageFolder(root=train_dir)
        self.tensor_transform = DATA_TRANSFORM
        self.recipe_ops = [build_recipe_augment(recipe) for recipe in top_recipes]
        self.task = task
        self.sensor = sensor
        self.class_counts = CLASS_AWARE_COUNTS[(task, sensor)]
        self.repeat_map = CLASS_AWARE_REPEAT_MAPS[(task, sensor)]
        self.virtual_index = []
        for base_idx, (path, y) in enumerate(self.base_ds.samples):
            class_name = self.base_ds.classes[int(y)]
            policy_class = resolve_policy_class(task, sensor, path, class_name)
            repeat_factor = int(self.repeat_map[policy_class])
            self.virtual_index.append((base_idx, "orig", None))
            for aug_idx in range(max(0, repeat_factor - 1)):
                recipe_idx = aug_idx % len(self.recipe_ops) if self.recipe_ops else None
                self.virtual_index.append((base_idx, "aug", recipe_idx))

    def __len__(self):
        return len(self.virtual_index)

    def __getitem__(self, idx):
        base_idx, mode, recipe_idx = self.virtual_index[idx]
        path, y = self.base_ds.samples[base_idx]
        image = self.base_ds.loader(path).convert("RGB")
        if mode == "aug" and recipe_idx is not None:
            image = self.recipe_ops[int(recipe_idx)](image)
        image = self.tensor_transform(image)
        return image, y, idx


def dataset_sample_records(train_ds):
    records = []
    for base_idx, _, _ in train_ds.virtual_index:
        path, y = train_ds.base_ds.samples[base_idx]
        records.append((path, train_ds.base_ds.classes[int(y)]))
    return records


def build_weighted_sampler(train_ds, task, sensor, gamma=SAMPLER_GAMMA):
    records = dataset_sample_records(train_ds)
    weight_keys = []
    for path, class_name in records:
        policy_class = resolve_policy_class(task, sensor, path, class_name)
        weight_keys.append(policy_class)
    key_counts = Counter(weight_keys)
    sample_weights = np.array([1.0 / (max(key_counts[key], 1) ** float(gamma)) for key in weight_keys], dtype=np.float64)
    sample_weights = sample_weights / sample_weights.mean()
    return WeightedRandomSampler(weights=torch.DoubleTensor(sample_weights.tolist()), num_samples=len(sample_weights), replacement=True)


In [ ]:
def cleanup_memory(*objs):
    for obj in objs:
        try:
            if hasattr(obj, "to"):
                obj.to("cpu")
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def create_model(model_name: str, num_classes: int):
    if model_name == "Resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )
        return model.to(DEVICE)

    if model_name == "ViT":
        try:
            import timm
        except Exception as error:
            raise ImportError("timm is required for ViT. Install it before running this notebook.") from error

        model = timm.create_model(
            "vit_base_patch32_224",
            pretrained=True,
            num_classes=num_classes,
        )
        return model.to(DEVICE)

    raise ValueError(f"Unsupported model: {model_name}")


sample_model_name = MODELS_TO_RUN[0]
sample_num_classes = EXPECTED_CLASSES[TASKS_TO_RUN[0]]
sample_model = create_model(sample_model_name, sample_num_classes)

print(f"Sample model: {sample_model_name}")
print(f"Number of classes: {sample_num_classes}")
print(f"Device: {DEVICE}")


In [ ]:
def compute_f1(y_true, y_pred, num_classes: int) -> float:
    if num_classes == 2:
        return float(f1_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0))
    return float(f1_score(y_true, y_pred, average="macro", zero_division=0))


def compute_auc(y_true, y_prob, num_classes: int) -> float:
    try:
        if num_classes == 2:
            return float(roc_auc_score(y_true, y_prob[:, 1]))
        return float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro"))
    except Exception:
        return float("nan")


def compute_tpr_fpr_by_class(y_true, y_pred, class_names):
    rows = []
    tpr_values = []
    fpr_values = []

    for idx, class_name in enumerate(class_names):
        y_true_bin = y_true == idx
        y_pred_bin = y_pred == idx

        tp = np.sum(y_true_bin & y_pred_bin)
        fn = np.sum(y_true_bin & (~y_pred_bin))
        fp = np.sum((~y_true_bin) & y_pred_bin)
        tn = np.sum((~y_true_bin) & (~y_pred_bin))

        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan

        rows.append({
            "class": class_name,
            "tpr": float(tpr) if not np.isnan(tpr) else np.nan,
            "fpr": float(fpr) if not np.isnan(fpr) else np.nan,
        })
        tpr_values.append(tpr)
        fpr_values.append(fpr)

    summary = {
        "avg_tpr": float(np.nanmean(tpr_values)) if len(tpr_values) else np.nan,
        "avg_fpr": float(np.nanmean(fpr_values)) if len(fpr_values) else np.nan,
    }
    return pd.DataFrame(rows), summary


def compute_eo_gap(y_true, y_pred, sensitive, num_classes: int, ignore_values=None):
    """Compute a multiclass Equalized Odds Gap using both TPR and FPR."""
    sensitive = np.asarray(sensitive)
    if ignore_values is not None:
        ignore_values = set(ignore_values)
        keep = np.asarray([value not in ignore_values for value in sensitive], dtype=bool)
        y_true = y_true[keep]
        y_pred = y_pred[keep]
        sensitive = sensitive[keep]

    groups = sorted(set(sensitive.tolist()))
    if len(groups) <= 1:
        return 0.0

    class_gaps = []
    for class_idx in range(num_classes):
        tprs = []
        fprs = []
        for group in groups:
            idx = sensitive == group
            yt = y_true[idx] == class_idx
            yp = y_pred[idx] == class_idx
            tp = np.sum(yt & yp)
            fn = np.sum(yt & (~yp))
            fp = np.sum((~yt) & yp)
            tn = np.sum((~yt) & (~yp))
            tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
            fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
            tprs.append(tpr)
            fprs.append(fpr)

        mean_tpr = np.nanmean(tprs)
        mean_fpr = np.nanmean(fprs)
        mean_tpr = 0.0 if np.isnan(mean_tpr) else mean_tpr
        mean_fpr = 0.0 if np.isnan(mean_fpr) else mean_fpr

        tpr_gaps = [abs((mean_tpr if np.isnan(tpr) else tpr) - mean_tpr) for tpr in tprs]
        fpr_gaps = [abs((mean_fpr if np.isnan(fpr) else fpr) - mean_fpr) for fpr in fprs]
        class_gap = 0.5 * (float(np.mean(tpr_gaps)) + float(np.mean(fpr_gaps)))
        class_gaps.append(class_gap)

    return float(np.mean(class_gaps)) if class_gaps else 0.0


def compute_G(eo_gap_age: float, eo_gap_sex: float, eo_gap_skin: float) -> float:
    return float(np.mean([
        1 - eo_gap_age,
        1 - eo_gap_sex,
        1 - eo_gap_skin,
    ]))


def compute_objective(auc: float, g_value: float, alpha: float = ALPHA) -> float:
    return float(alpha * auc + (1 - alpha) * g_value)


def get_predictions_targets_probs(model, loader):
    model.eval()
    y_true_all, y_pred_all, y_prob_all = [], [], []
    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(DEVICE)
            logits = model(images)
            y_true_all.append(labels.numpy())
            y_pred_all.append(logits.argmax(dim=1).cpu().numpy())
            y_prob_all.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(y_true_all), np.concatenate(y_pred_all), np.concatenate(y_prob_all)


def age_sort_key(value):
    if value == "unknown":
        return (1, float("inf"))
    return (0, int(value))


def build_sensitive_arrays(dataset):
    skin_vals, sex_vals, age_vals = [], [], []
    for path, _ in dataset.samples:
        isic_id = Path(path).stem
        skin_vals.append(int(isic_to_skin.get(isic_id, -1)))
        sex_vals.append(SEX_TO_ID[normalize_sex(isic_to_sex.get(isic_id, "unknown"))])
        age_vals.append(normalize_age(isic_to_age.get(isic_id, "unknown")))

    unique_age_groups = sorted(set(age_vals), key=age_sort_key)
    age_map = {value: idx for idx, value in enumerate(unique_age_groups)}
    age_ids = np.array([age_map[value] for value in age_vals], dtype=np.int32)

    return np.array(skin_vals, dtype=np.int32), np.array(sex_vals, dtype=np.int32), age_ids


def evaluate_split(model, loader, dataset, class_names):
    y_true, y_pred, y_prob = get_predictions_targets_probs(model, loader)
    num_classes = len(class_names)
    skin_arr, sex_arr, age_arr = build_sensitive_arrays(dataset)

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": compute_f1(y_true, y_pred, num_classes),
        "auc": compute_auc(y_true, y_prob, num_classes),
    }
    metrics["eo_gap_skin"] = compute_eo_gap(y_true, y_pred, skin_arr, num_classes, ignore_values={0, -1})
    metrics["eo_gap_sex"] = compute_eo_gap(y_true, y_pred, sex_arr, num_classes)
    metrics["eo_gap_age"] = compute_eo_gap(y_true, y_pred, age_arr, num_classes)
    metrics["G"] = compute_G(
        metrics["eo_gap_age"],
        metrics["eo_gap_sex"],
        metrics["eo_gap_skin"],
    )
    metrics["objective"] = compute_objective(metrics["auc"], metrics["G"], ALPHA)

    tpr_fpr_df, tpr_fpr_summary = compute_tpr_fpr_by_class(y_true, y_pred, class_names)
    metrics["avg_tpr"] = tpr_fpr_summary["avg_tpr"]
    metrics["avg_fpr"] = tpr_fpr_summary["avg_fpr"]

    per_class_rows = []
    for idx, class_name in enumerate(class_names):
        mask = y_true == idx
        class_acc = float(accuracy_score(y_true[mask], y_pred[mask])) if np.sum(mask) else np.nan
        per_class_rows.append({"class": class_name, "n": int(np.sum(mask)), "accuracy": class_acc})
        metrics[f"acc_{safe_name(class_name)}"] = class_acc

    per_class_df = pd.DataFrame(per_class_rows).merge(tpr_fpr_df, on="class", how="left")

    for _, row in tpr_fpr_df.iterrows():
        class_key = safe_name(row["class"])
        metrics[f"tpr_{class_key}"] = row["tpr"]
        metrics[f"fpr_{class_key}"] = row["fpr"]

    return metrics, per_class_df


def train_step(model, dataloader, loss_fn, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / max(total, 1), correct / max(total, 1)


def val_step(model, dataloader, loss_fn, num_classes: int):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    y_true_all = []
    y_prob_all = []
    with torch.inference_mode():
        for images, labels in dataloader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(images)
            loss = loss_fn(logits, labels)
            running_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            y_true_all.append(labels.cpu().numpy())
            y_prob_all.append(torch.softmax(logits, dim=1).cpu().numpy())
    y_true = np.concatenate(y_true_all) if y_true_all else np.array([])
    y_prob = np.concatenate(y_prob_all) if y_prob_all else np.array([])
    val_auc = compute_auc(y_true, y_prob, num_classes) if y_true.size else float("nan")
    return running_loss / max(total, 1), correct / max(total, 1), val_auc


def train_model(model, train_loader, val_loader, optimizer, loss_fn, epochs, patience, num_classes: int):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_auc": []}
    best_val_auc = -np.inf
    best_state = copy.deepcopy(model.state_dict())
    patience_count = 0
    for epoch in tqdm(range(epochs), leave=False):
        train_loss, train_acc = train_step(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc, val_auc = val_step(model, val_loader, loss_fn, num_classes)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_auc"].append(val_auc)
        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_auc={val_auc:.4f}"
        )
        improved = val_auc > best_val_auc if not np.isnan(val_auc) else False
        if improved:
            print(" Validation AUC improved, saving best model...")
            best_val_auc = val_auc
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break
    model.load_state_dict(best_state)
    return history


def append_row(csv_path, row):
    old_df = read_csv_if_exists(csv_path)
    new_df = pd.DataFrame([row])
    if old_df.empty:
        out_df = new_df
    else:
        cols = list(dict.fromkeys(list(old_df.columns) + list(new_df.columns)))
        out_df = pd.concat([old_df.reindex(columns=cols), new_df.reindex(columns=cols)], ignore_index=True)
    out_df.to_csv(csv_path, index=False, lineterminator="\n")


def read_csv_if_exists(csv_path):
    if not csv_path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(csv_path)
    except Exception:
        return pd.DataFrame()


def compute_averages_from_experiments(exp_df):
    if exp_df.empty:
        return pd.DataFrame()

    ok_df = exp_df[exp_df["status"] == "ok"].copy()
    if ok_df.empty:
        return pd.DataFrame()

    group_cols = ["model", "task", "sensor", "hp_tag", "lr", "batch_size"]
    excluded = {"run_id", "seed", "status", "model_path", "report_path", "best_epoch", "best_val_auc", "error"}
    numeric_metric_cols = []
    passthrough_cols = []

    for col in ok_df.columns:
        if col in group_cols or col in excluded:
            continue
        converted = pd.to_numeric(ok_df[col], errors="coerce")
        if converted.notna().any():
            ok_df[col] = converted
            numeric_metric_cols.append(col)
        else:
            passthrough_cols.append(col)

    avg_df = ok_df.groupby(group_cols).agg(
        n_runs_completed=("run_id", "count"),
        **{col: (col, "mean") for col in numeric_metric_cols},
    ).reset_index()

    for col in passthrough_cols:
        first_values = ok_df.groupby(group_cols)[col].first().reset_index(name=col)
        avg_df = avg_df.merge(first_values, on=group_cols, how="left")

    return avg_df


def df_to_md(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)


def write_run_report(path, run_id, model_name, task, sensor, lr, batch_size, seed, best_epoch, best_val_auc, model_path, val_metrics, test_metrics, val_class_df, test_class_df):
    global_df = pd.DataFrame([
        {
            "Split": "Validation",
            "Accuracy": val_metrics["accuracy"],
            "F1": val_metrics["f1"],
            "AUC": val_metrics["auc"],
            "Average TPR": val_metrics["avg_tpr"],
            "Average FPR": val_metrics["avg_fpr"],
            "G": val_metrics["G"],
            "Objective": val_metrics["objective"],
            "Equalized Odds Gap (skin)": val_metrics["eo_gap_skin"],
            "Equalized Odds Gap (sex)": val_metrics["eo_gap_sex"],
            "Equalized Odds Gap (age)": val_metrics["eo_gap_age"],
        },
        {
            "Split": "Test",
            "Accuracy": test_metrics["accuracy"],
            "F1": test_metrics["f1"],
            "AUC": test_metrics["auc"],
            "Average TPR": test_metrics["avg_tpr"],
            "Average FPR": test_metrics["avg_fpr"],
            "G": test_metrics["G"],
            "Objective": test_metrics["objective"],
            "Equalized Odds Gap (skin)": test_metrics["eo_gap_skin"],
            "Equalized Odds Gap (sex)": test_metrics["eo_gap_sex"],
            "Equalized Odds Gap (age)": test_metrics["eo_gap_age"],
        },
    ])

    text = f"""# Run {run_id}

## Hyperparameters
- **model**: {model_name}
- **task**: {task}
- **sensor**: {sensor}
- **learning_rate**: {lr}
- **batch_size**: {batch_size}
- **alpha**: {ALPHA}
- **seed**: {seed}
- **best_epoch**: {best_epoch}
- **best_val_auc**: {best_val_auc:.6f}
- **model_path**: {model_path}

## Global metrics
{df_to_md(global_df)}

## Validation per-class metrics
{df_to_md(val_class_df)}

## Test per-class metrics
{df_to_md(test_class_df)}
"""
    path.write_text(text, encoding="utf-8")


def write_summary_report(path, model_name, task, sensor, hp_tag, lr, batch_size, avg_row):
    metric_rows = []
    for metric in ["accuracy", "f1", "auc", "avg_tpr", "avg_fpr", "G", "objective", "eo_gap_skin", "eo_gap_sex", "eo_gap_age"]:
        metric_rows.append({
            "Metric": metric,
            "Validation Mean": float(avg_row.get(f"val_{metric}", np.nan)),
            "Test Mean": float(avg_row.get(f"test_{metric}", np.nan)),
        })

    class_keys = sorted({
        col.replace("val_acc_", "", 1)
        for col in avg_row.index
        if col.startswith("val_acc_")
    })
    val_class_rows = [
        {
            "class": class_key,
            "accuracy_mean": float(avg_row.get(f"val_acc_{class_key}", np.nan)),
            "tpr_mean": float(avg_row.get(f"val_tpr_{class_key}", np.nan)),
            "fpr_mean": float(avg_row.get(f"val_fpr_{class_key}", np.nan)),
        }
        for class_key in class_keys
    ]
    test_class_rows = [
        {
            "class": class_key,
            "accuracy_mean": float(avg_row.get(f"test_acc_{class_key}", np.nan)),
            "tpr_mean": float(avg_row.get(f"test_tpr_{class_key}", np.nan)),
            "fpr_mean": float(avg_row.get(f"test_fpr_{class_key}", np.nan)),
        }
        for class_key in class_keys
    ]

    text = f"""# Summary {hp_tag}

## Hyperparameters
- **model**: {model_name}
- **task**: {task}
- **sensor**: {sensor}
- **learning_rate**: {lr}
- **batch_size**: {batch_size}
- **alpha**: {ALPHA}
- **n_runs_completed**: {int(avg_row['n_runs_completed'])}

## Global metrics (mean)
{df_to_md(pd.DataFrame(metric_rows))}

## Validation per-class metrics (mean)
{df_to_md(pd.DataFrame(val_class_rows))}

## Test per-class metrics (mean)
{df_to_md(pd.DataFrame(test_class_rows))}
"""
    path.write_text(text, encoding="utf-8")


In [ ]:
class SampleWeightedCrossEntropy(nn.Module):
    def __init__(self, sample_weights):
        super().__init__()
        self.register_buffer("sample_weights", torch.tensor(sample_weights, dtype=torch.float32))

    def forward(self, logits, targets, sample_indices=None):
        if sample_indices is None:
            return F.cross_entropy(logits, targets)
        losses = F.cross_entropy(logits, targets, reduction="none")
        weights = self.sample_weights[sample_indices]
        return (losses * weights).mean()


def compute_virtual_class_skin_weights(train_ds, beta: float):
    records = dataset_sample_records(train_ds)
    pair_keys = []
    for path, class_name in records:
        policy_class = resolve_policy_class(train_ds.task, train_ds.sensor, path, class_name)
        skin_tone = int(isic_to_skin.get(extract_source_stem(path), -1))
        pair_keys.append((policy_class, skin_tone))

    valid_pair_keys = [key for key in pair_keys if int(key[1]) > 0]
    pair_counts = Counter(valid_pair_keys)
    raw_weights = {}
    for key, count in pair_counts.items():
        effective_num = 1.0 - np.power(beta, max(int(count), 1))
        raw_weights[key] = (1.0 - beta) / max(effective_num, 1e-12)

    normalization = np.mean(list(raw_weights.values())) if raw_weights else 1.0
    normalization = 1.0 if normalization == 0 else normalization
    normalized_weights = {key: float(value / normalization) for key, value in raw_weights.items()}
    sample_weights = np.array([normalized_weights.get(key, 0.0) for key in pair_keys], dtype=np.float32)

    pair_df = pd.DataFrame([
        {"class": key[0], "skin_tone": int(key[1]), "pair_count": int(pair_counts[key]), "pair_weight": float(normalized_weights[key])}
        for key in sorted(pair_counts.keys(), key=lambda item: (str(item[0]), int(item[1])))
    ])
    return sample_weights, pair_df

def create_custom_loss_for_virtual_dataset(train_ds, beta: float = DEFAULT_CUSTOM_LOSS_BETA):
    sample_weights, pair_df = compute_virtual_class_skin_weights(train_ds, beta=beta)
    return SampleWeightedCrossEntropy(sample_weights).to(DEVICE), sample_weights, pair_df


def train_step(model, dataloader, loss_fn, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for batch in dataloader:
        if len(batch) == 3:
            images, labels, sample_indices = batch
        else:
            images, labels = batch
            sample_indices = None
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        if sample_indices is not None:
            sample_indices = sample_indices.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels, sample_indices) if sample_indices is not None else loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / max(total, 1), correct / max(total, 1)

In [ ]:
def phase_roots(direction: str, phase: str):
    return (
        MODELS_ROOT / direction / phase,
        REPORTS_ROOT / direction / phase,
        RUNS_ROOT / direction / phase,
    )


def load_state_dict_flexible(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    state_dict = checkpoint.get("state_dict", checkpoint)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing or unexpected:
        print(f"Loaded checkpoint with missing keys={len(missing)} unexpected keys={len(unexpected)}")
    return checkpoint


def run_supervised_aug_custom_phase(
    *,
    phase,
    direction,
    task,
    sensor,
    model_name,
    lr,
    batch_size,
    beta,
    run_idx,
    source_sensor,
    target_sensor,
    source_checkpoint=None,
):
    train_data, val_data, test_data, class_names = load_datasets(task, sensor)
    top_recipes = TOP10_RECIPES_BY_TASK_SENSOR[(task, sensor)]
    val_loader, test_loader = make_eval_dataloaders(val_data, test_data, batch_size=batch_size)

    models_root, reports_root, runs_root = phase_roots(direction, phase)
    combo_models_root = models_root / model_name / task / sensor
    combo_reports_root = reports_root / model_name / task / sensor
    combo_runs_root = runs_root / model_name / task / sensor
    for path_obj in [combo_models_root, combo_reports_root, combo_runs_root]:
        path_obj.mkdir(parents=True, exist_ok=True)

    hp_tag = f"lr{format_lr(lr)}_bs{batch_size}_beta{format_float_tag(beta)}"
    hp_model_dir = combo_models_root / hp_tag
    hp_report_dir = combo_reports_root / hp_tag
    hp_model_dir.mkdir(parents=True, exist_ok=True)
    hp_report_dir.mkdir(parents=True, exist_ok=True)

    phase_tag = "src" if phase == "source_training" else "tgt"
    direction_tag = "c2d" if direction == "clinical_to_dermoscopic" else "d2c"
    run_seed = BASE_SEED + run_idx
    run_id = f"{model_name}_{task}_{direction_tag}_{phase_tag}_{hp_tag}_r{run_idx:02d}"
    model_path = hp_model_dir / f"{run_id}.pth"
    report_path = hp_report_dir / f"{run_id}.md"
    history_path = hp_report_dir / f"r{run_idx:02d}_history.csv"
    loss_weights_path = hp_report_dir / f"r{run_idx:02d}_loss_weights.csv"
    experiments_csv = combo_runs_root / "experiments.csv"
    averages_csv = combo_runs_root / "averages.csv"

    exp_df = read_csv_if_exists(experiments_csv)
    if RESUME and not exp_df.empty and "run_id" in exp_df.columns:
        already_ok = ((exp_df["run_id"] == run_id) & (exp_df["status"] == "ok")).any()
        if already_ok and model_path.exists():
            print(f"Skipping: {run_id}")
            return model_path

    set_all_seeds(run_seed)
    model = None
    train_loader = None
    train_ds = None
    try:
        train_ds = OnlineClassAwareTop10Dataset(train_dir=DATASET_ROOTS[(task, sensor)] / "train", top_recipes=top_recipes, task=task, sensor=sensor)
        sampler = build_weighted_sampler(train_ds, task=task, sensor=sensor)
        train_loader = DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=False,
            sampler=sampler,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
            persistent_workers=NUM_WORKERS > 0,
        )
        loss_fn, sample_weights, pair_df = create_custom_loss_for_virtual_dataset(train_ds, beta=beta)
        pair_df.to_csv(loss_weights_path, index=False)

        model = create_model(model_name, num_classes=len(class_names))
        if source_checkpoint is not None:
            load_state_dict_flexible(model, source_checkpoint)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

        history = train_model(
            model,
            train_loader,
            val_loader,
            optimizer,
            loss_fn,
            EPOCHS,
            EARLY_STOPPING_PATIENCE_BY_MODEL[model_name],
            num_classes=len(class_names),
        )
        pd.DataFrame(history).to_csv(history_path, index=False)
        val_metrics, val_class_df = evaluate_split(model, val_loader, val_data, class_names)
        test_metrics, test_class_df = evaluate_split(model, test_loader, test_data, class_names)
        best_epoch = int(np.nanargmax(history["val_auc"])) + 1 if history["val_auc"] else 0
        best_val_auc = float(np.nanmax(history["val_auc"])) if history["val_auc"] else np.nan

        torch.save(
            {
                "state_dict": model.state_dict(),
                "model_name": model_name,
                "task": task,
                "sensor": sensor,
                "phase": phase,
                "direction": direction,
                "source_sensor": source_sensor,
                "target_sensor": target_sensor,
                "source_checkpoint": str(source_checkpoint) if source_checkpoint else "",
                "num_classes": len(class_names),
                "class_names": class_names,
                "lr": lr,
                "batch_size": batch_size,
                "beta": beta,
                "run_id": run_id,
                "selected_recipes": [recipe["recipe_name"] for recipe in top_recipes],
                "sampler_gamma": SAMPLER_GAMMA,
                "sampler_key": "class",
                "augmentation_policy": "class_aware_repeat_factor",
                "loss_policy": "sample_weighted_cross_entropy_class_x_skin_tone",
            },
            model_path,
        )

        row = {
            "model": model_name,
            "task": task,
            "sensor": sensor,
            "hp_tag": hp_tag,
            "run_id": run_id,
            "seed": run_seed,
            "lr": lr,
            "batch_size": batch_size,
            "beta": beta,
            "status": "ok",
            "model_path": str(model_path),
            "report_path": str(report_path),
            "best_epoch": best_epoch,
            "best_val_auc": best_val_auc,
            "n_train_samples": len(train_ds),
            "sampler_gamma": SAMPLER_GAMMA,
            "sampler_key": "class",
            "augmentation_policy": "class_aware_repeat_factor",
            "loss_policy": "sample_weighted_cross_entropy_class_x_skin_tone",
            "val_accuracy": val_metrics["accuracy"],
            "val_f1": val_metrics["f1"],
            "val_auc": val_metrics["auc"],
            "val_avg_tpr": val_metrics["avg_tpr"],
            "val_avg_fpr": val_metrics["avg_fpr"],
            "val_G": val_metrics["G"],
            "val_objective": val_metrics["objective"],
            "val_eo_gap_skin": val_metrics["eo_gap_skin"],
            "val_eo_gap_sex": val_metrics["eo_gap_sex"],
            "val_eo_gap_age": val_metrics["eo_gap_age"],
            "test_accuracy": test_metrics["accuracy"],
            "test_f1": test_metrics["f1"],
            "test_auc": test_metrics["auc"],
            "test_avg_tpr": test_metrics["avg_tpr"],
            "test_avg_fpr": test_metrics["avg_fpr"],
            "test_G": test_metrics["G"],
            "test_objective": test_metrics["objective"],
            "test_eo_gap_skin": test_metrics["eo_gap_skin"],
            "test_eo_gap_sex": test_metrics["eo_gap_sex"],
            "test_eo_gap_age": test_metrics["eo_gap_age"],
        }
        for key_name, value in val_metrics.items():
            if key_name.startswith("acc_") or key_name.startswith("tpr_") or key_name.startswith("fpr_"):
                row[f"val_{key_name}"] = value
        for key_name, value in test_metrics.items():
            if key_name.startswith("acc_") or key_name.startswith("tpr_") or key_name.startswith("fpr_"):
                row[f"test_{key_name}"] = value

        append_row(experiments_csv, row)
        write_run_report(report_path, run_id, model_name, task, sensor, lr, batch_size, run_seed, best_epoch, best_val_auc, model_path, val_metrics, test_metrics, val_class_df, test_class_df)
        avg_df = compute_averages_from_experiments(read_csv_if_exists(experiments_csv))
        if not avg_df.empty:
            avg_df.to_csv(averages_csv, index=False)
            hp_row = avg_df[avg_df["hp_tag"] == hp_tag]
            if not hp_row.empty:
                write_summary_report(hp_report_dir / "summary.md", model_name, task, sensor, hp_tag, lr, batch_size, hp_row.iloc[0])

        print(f"{run_id} | val_auc={val_metrics['auc']:.4f} test_auc={test_metrics['auc']:.4f} test_objective={test_metrics['objective']:.4f}")
        return model_path
    except Exception as error:
        append_row(experiments_csv, {"model": model_name, "task": task, "sensor": sensor, "hp_tag": hp_tag, "run_id": run_id, "seed": run_seed, "lr": lr, "batch_size": batch_size, "beta": beta, "status": "fail", "error": repr(error)})
        print(f"Failed: {run_id} -> {error}")
        return None
    finally:
        cleanup_memory(model, train_loader, train_ds)

In [ ]:
transfer_index_rows = []

for task in TASKS_TO_RUN:
    cfg = WINNING_TRANSFER_FULL_CONFIGS[task]
    model_name = cfg["model"]
    lr = float(cfg["lr"])
    batch_size = int(cfg["batch_size"])
    beta = float(cfg["beta"])
    n_runs = N_RUNS_BY_MODEL[model_name]

    for transfer in TRANSFER_DIRECTIONS:
        source_sensor = transfer["source_sensor"]
        target_sensor = transfer["target_sensor"]
        direction = transfer["direction"]
        print("\n" + "=" * 100)
        print(f"{task} | {direction} | online augmentation + custom loss | lr={lr} | batch_size={batch_size} | beta={beta} | runs={n_runs}")
        print("=" * 100)
        for run_idx in range(1, n_runs + 1):
            source_checkpoint = run_supervised_aug_custom_phase(
                phase="source_training",
                direction=direction,
                task=task,
                sensor=source_sensor,
                model_name=model_name,
                lr=lr,
                batch_size=batch_size,
                beta=beta,
                run_idx=run_idx,
                source_sensor=source_sensor,
                target_sensor=target_sensor,
            )
            if source_checkpoint is None:
                continue
            target_checkpoint = run_supervised_aug_custom_phase(
                phase="target_finetuning",
                direction=direction,
                task=task,
                sensor=target_sensor,
                model_name=model_name,
                lr=lr,
                batch_size=batch_size,
                beta=beta,
                run_idx=run_idx,
                source_sensor=source_sensor,
                target_sensor=target_sensor,
                source_checkpoint=source_checkpoint,
            )
            transfer_index_rows.append({
                "task": task,
                "direction": direction,
                "source_sensor": source_sensor,
                "target_sensor": target_sensor,
                "model": model_name,
                "run_idx": run_idx,
                "lr": lr,
                "batch_size": batch_size,
                "beta": beta,
                "source_checkpoint": str(source_checkpoint),
                "target_checkpoint": str(target_checkpoint) if target_checkpoint else "",
            })
            pd.DataFrame(transfer_index_rows).to_csv(RUNS_ROOT / "transfer_run_index.csv", index=False)

print("Online augmentation + custom-loss transfer-learning pipeline finished.")

In [ ]:
def build_top5_table(avg_df, top_k=5):
    if avg_df.empty:
        return pd.DataFrame()
    sort_cols = [col for col in ["test_objective", "test_auc", "val_objective"] if col in avg_df.columns]
    if not sort_cols:
        return pd.DataFrame()
    top_df = avg_df.sort_values(sort_cols, ascending=[False] * len(sort_cols)).head(top_k).copy()
    core_cols = [
        "model", "task", "sensor", "hp_tag", "lr", "batch_size", "beta", "n_runs_completed",
        "val_accuracy", "val_f1", "val_auc", "val_avg_tpr", "val_avg_fpr", "val_G", "val_objective", "val_eo_gap_skin", "val_eo_gap_sex", "val_eo_gap_age",
        "test_accuracy", "test_f1", "test_auc", "test_avg_tpr", "test_avg_fpr", "test_G", "test_objective", "test_eo_gap_skin", "test_eo_gap_sex", "test_eo_gap_age",
    ]
    class_cols = sorted([col for col in top_df.columns if col.startswith("val_acc_") or col.startswith("test_acc_") or col.startswith("val_tpr_") or col.startswith("test_tpr_") or col.startswith("val_fpr_") or col.startswith("test_fpr_")])
    cols = [col for col in core_cols + class_cols if col in top_df.columns]
    return top_df[cols]


updated = 0
for experiments_csv in sorted(RUNS_ROOT.rglob("experiments.csv")):
    run_dir = experiments_csv.parent
    averages_csv = run_dir / "averages.csv"
    top5_csv = run_dir / "top5_summary.csv"
    top5_md = run_dir / "top5_summary.md"
    exp_df = read_csv_if_exists(experiments_csv)
    if exp_df.empty:
        continue
    avg_df = compute_averages_from_experiments(exp_df)
    if avg_df.empty:
        continue
    avg_df.to_csv(averages_csv, index=False)
    top5_df = build_top5_table(avg_df, top_k=5)
    top5_df.to_csv(top5_csv, index=False)
    md_text = f"""# Top-5 Summary

- Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
- Source: {averages_csv}
- Ranking: test_objective, test_auc, val_objective

{df_to_md(top5_df)}
"""
    top5_md.write_text(md_text, encoding="utf-8")
    updated += 1
    print(f"Updated: {run_dir}")

print(f"Final consolidation complete. Updated folders: {updated}")

In [ ]:
BASELINE_RUNS_ROOT = (PROJECT_ROOT if "PROJECT_ROOT" in globals() else Path.cwd()) / "Experiment_1" / "training_runs_final_fixed" / "runs"


def _to_float(value):
    try:
        return float(value)
    except Exception:
        return np.nan


def _target_average_rows():
    rows = []
    for transfer in TRANSFER_DIRECTIONS:
        direction = transfer["direction"]
        source_sensor = transfer["source_sensor"]
        target_sensor = transfer["target_sensor"]
        for task in TASKS_TO_RUN:
            avg_path = RUNS_ROOT / direction / "target_finetuning" / "ViT" / task / target_sensor / "averages.csv"
            avg_df = read_csv_if_exists(avg_path)
            if avg_df.empty:
                continue
            for col in ["lr", "batch_size", "beta", "val_objective", "test_objective", "val_auc", "test_auc", "val_accuracy", "test_accuracy"]:
                if col in avg_df.columns:
                    avg_df[col] = pd.to_numeric(avg_df[col], errors="coerce")
            avg_df = avg_df.copy()
            avg_df["direction"] = direction
            avg_df["source_sensor"] = source_sensor
            avg_df["target_sensor"] = target_sensor
            avg_df["averages_csv"] = str(avg_path)
            rows.append(avg_df)
    return pd.concat(rows, ignore_index=True, sort=False) if rows else pd.DataFrame()


def _select_best_by_validation_objective(results_df):
    if results_df.empty:
        return results_df
    selected = []
    group_cols = ["task", "target_sensor", "direction"]
    for _, group_df in results_df.groupby(group_cols, sort=False):
        sort_cols = [col for col in ["val_objective", "test_objective", "val_auc"] if col in group_df.columns]
        selected.append(group_df.sort_values(sort_cols, ascending=[False] * len(sort_cols)).iloc[0])
    return pd.DataFrame(selected).reset_index(drop=True)


def _global_split_row(row, split):
    split_label = "Validation" if split == "val" else "Test"
    out = {
        "Task": row["task"],
        "Target Sensor": row["target_sensor"],
        "Source Sensor": row["source_sensor"],
        "Direction": row["direction"],
        "Model": row["model"],
        "HP Tag": row["hp_tag"],
        "LR": _to_float(row.get("lr", np.nan)),
        "Batch Size": int(row["batch_size"]) if pd.notna(row.get("batch_size", np.nan)) else np.nan,
        "Split": split_label,
        "Accuracy": _to_float(row.get(f"{split}_accuracy", np.nan)),
        "F1": _to_float(row.get(f"{split}_f1", np.nan)),
        "AUC": _to_float(row.get(f"{split}_auc", np.nan)),
        "Average TPR": _to_float(row.get(f"{split}_avg_tpr", np.nan)),
        "Average FPR": _to_float(row.get(f"{split}_avg_fpr", np.nan)),
        "G": _to_float(row.get(f"{split}_G", np.nan)),
        "Objective": _to_float(row.get(f"{split}_objective", np.nan)),
        "EO Gap Skin": _to_float(row.get(f"{split}_eo_gap_skin", np.nan)),
        "EO Gap Sex": _to_float(row.get(f"{split}_eo_gap_sex", np.nan)),
        "EO Gap Age": _to_float(row.get(f"{split}_eo_gap_age", np.nan)),
        "Averages CSV": row.get("averages_csv", ""),
    }
    if "beta" in row.index and pd.notna(row.get("beta", np.nan)):
        out["Beta"] = _to_float(row.get("beta", np.nan))
    return out


def build_best_model_per_task_sensor_table(best_df):
    rows = []
    for _, row in best_df.iterrows():
        rows.append(_global_split_row(row, "val"))
        rows.append(_global_split_row(row, "test"))
    return pd.DataFrame(rows)


def _load_baseline_best_row(task, sensor, model="ViT"):
    avg_path = BASELINE_RUNS_ROOT / model / task / sensor / "averages.csv"
    avg_df = read_csv_if_exists(avg_path)
    if avg_df.empty:
        return None
    for col in ["lr", "batch_size", "val_objective", "test_objective", "val_auc", "test_auc"]:
        if col in avg_df.columns:
            avg_df[col] = pd.to_numeric(avg_df[col], errors="coerce")
    sort_cols = [col for col in ["val_objective", "test_objective", "val_auc"] if col in avg_df.columns]
    return avg_df.sort_values(sort_cols, ascending=[False] * len(sort_cols)).iloc[0]


def _class_supports(task, sensor, split):
    split_dir = "validation" if split == "val" else "test"
    ds = ImageFolder(DATASET_ROOTS[(task, sensor)] / split_dir)
    support = {}
    for class_idx, class_name in enumerate(ds.classes):
        support[safe_name(class_name)] = int(sum(1 for target in ds.targets if target == class_idx))
    return support, int(len(ds)), {safe_name(name): name for name in ds.classes}


def _derive_class_metrics(row, split, suffix, support, total):
    acc = _to_float(row.get(f"{split}_acc_{suffix}", np.nan))
    recall = _to_float(row.get(f"{split}_tpr_{suffix}", np.nan))
    fpr = _to_float(row.get(f"{split}_fpr_{suffix}", np.nan))
    n_pos = int(support.get(suffix, 0))
    n_neg = max(int(total) - n_pos, 0)
    tp = recall * n_pos if pd.notna(recall) else np.nan
    fp = fpr * n_neg if pd.notna(fpr) else np.nan
    precision = tp / (tp + fp) if pd.notna(tp) and pd.notna(fp) and (tp + fp) > 0 else np.nan
    f1 = (2 * precision * recall / (precision + recall)) if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0 else np.nan
    return {"accuracy": acc, "recall": recall, "precision": precision, "f1": f1}


def build_per_class_comparison_table(best_df):
    rows = []
    for _, transfer_row in best_df.iterrows():
        task = transfer_row["task"]
        target_sensor = transfer_row["target_sensor"]
        model = transfer_row["model"]
        baseline_row = _load_baseline_best_row(task, target_sensor, model=model)
        if baseline_row is None:
            continue
        for split in ["val", "test"]:
            support, total, label_map = _class_supports(task, target_sensor, split)
            for suffix, class_label in label_map.items():
                transfer_metrics = _derive_class_metrics(transfer_row, split, suffix, support, total)
                baseline_metrics = _derive_class_metrics(baseline_row, split, suffix, support, total)
                rows.append({
                    "Task": task,
                    "Target Sensor": target_sensor,
                    "Source Sensor": transfer_row["source_sensor"],
                    "Direction": transfer_row["direction"],
                    "Split": "Validation" if split == "val" else "Test",
                    "Class": class_label,
                    "Support": support.get(suffix, 0),
                    "Transfer Accuracy": transfer_metrics["accuracy"],
                    "Baseline Accuracy": baseline_metrics["accuracy"],
                    "Delta Accuracy": transfer_metrics["accuracy"] - baseline_metrics["accuracy"],
                    "Transfer Recall": transfer_metrics["recall"],
                    "Baseline Recall": baseline_metrics["recall"],
                    "Delta Recall": transfer_metrics["recall"] - baseline_metrics["recall"],
                    "Transfer Precision": transfer_metrics["precision"],
                    "Baseline Precision": baseline_metrics["precision"],
                    "Delta Precision": transfer_metrics["precision"] - baseline_metrics["precision"],
                    "Transfer F1": transfer_metrics["f1"],
                    "Baseline F1": baseline_metrics["f1"],
                    "Delta F1": transfer_metrics["f1"] - baseline_metrics["f1"],
                })
    return pd.DataFrame(rows)


all_target_finetuning_results = _target_average_rows()
best_target_finetuning_rows = _select_best_by_validation_objective(all_target_finetuning_results)

best_model_per_task_sensor_table = build_best_model_per_task_sensor_table(best_target_finetuning_rows)
per_class_comparison_table = build_per_class_comparison_table(best_target_finetuning_rows)

summary_out = RUNS_ROOT / "best_model_per_task_sensor_selected_by_validation_objective.csv"
per_class_out = RUNS_ROOT / "per_class_comparison_vs_baseline.csv"
long_summary_out = RUNS_ROOT / "final_target_finetuning_summary_long.csv"

if not best_model_per_task_sensor_table.empty:
    best_model_per_task_sensor_table.to_csv(summary_out, index=False)
    best_model_per_task_sensor_table.to_csv(long_summary_out, index=False)
if not per_class_comparison_table.empty:
    per_class_comparison_table.to_csv(per_class_out, index=False)
    for direction, direction_df in per_class_comparison_table.groupby("Direction", sort=False):
        direction_slug = safe_name(direction)
        direction_df.to_csv(RUNS_ROOT / f"per_class_comparison_vs_baseline_{direction_slug}.csv", index=False)

print("Best model per task and target sensor selected by validation objective")
display(best_model_per_task_sensor_table)

if per_class_comparison_table.empty:
    print("Per-class comparison against Experiment 1 baseline: no rows available")
else:
    for direction, direction_df in per_class_comparison_table.groupby("Direction", sort=False):
        print(f"Per-class comparison against Experiment 1 baseline | {direction}")
        display(direction_df.reset_index(drop=True))
